## Pipeline推理

In [1]:
from transformers import pipeline

generator = pipeline("text-generation", model="/mnt/chinese-alpaca-2-7b")
outputs = generator("What is the capital of Australia?", return_full_text=False, do_sample=False, max_new_tokens=100)
print(outputs[0]['generated_text'])

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

 澳大利亚的首都是悉尼。悉尼是澳大利亚最大的城市，也是澳大利亚的经济、文化、交通中心。悉尼是澳大利亚的金融中心，也是澳大利亚最大的港口。悉尼是澳大利亚的旅游中心，每年有大量的游客到悉尼观光。悉尼是澳大利亚的文化中心，悉尼歌剧院是悉尼的标志。悉尼是澳大利亚的教育中心，悉尼大学是澳大利亚最好的大学。悉尼是澳


In [8]:
outputs = generator("上海有什么好吃的小吃?", return_full_text=False, max_new_tokens=500)
outputs

[{'generated_text': ' 上海有很多好吃的小吃，以下是一些推荐：\n1. 生煎包：上海的生煎包是著名的特色小吃之一，外皮酥脆，内馅鲜美。 2. 小笼包：小笼包是上海的传统点心之一，外皮薄而有弹性，内馅鲜美。 3. 蟹粉小笼包：蟹粉小笼包是上海的特色小吃之一，外皮薄而有弹性，内馅鲜美，加入蟹粉的鲜味。 4. 糯米鸡：糯米鸡是上海的传统小吃之一，外皮酥脆，内馅鲜美。 5. 红烧肉：红烧肉是上海的传统菜肴之一，肉质鲜嫩，味道浓郁。 6. 油条：油条是上海的传统小吃之一，外皮酥脆，内馅鲜美。 7. 糖醋排骨：糖醋排骨是上海的传统菜肴之一，肉质鲜嫩，味道酸甜可口。 8. 麻球：麻球是上海的传统小吃之一，外皮酥脆，内馅鲜美。 9. 豆腐脑：豆腐脑是上海的传统小吃之一，口感细腻，味道鲜美。 10. 红烧鱼：红烧鱼是上海的传统菜肴之一，鱼肉鲜嫩，味道浓郁。'}]

In [6]:
outputs = generator("What is the capital of Australia?", return_tensors = True)
outputs

[{'generated_token_ids': [1724,
   338,
   278,
   7483,
   310,
   8314,
   29973,
   1815,
   495,
   336,
   29889,
   1815,
   495,
   336,
   338,
   278,
   7483,
   310,
   8314,
   29889]}]

## Model and Tokenizer API推理

In [17]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("/mnt/chinese-alpaca-2-7b")
model = AutoModelForCausalLM.from_pretrained("/mnt/chinese-alpaca-2-7b")

inputs = tokenizer("What is the capital of Australia?", return_tensors="pt")
outputs = model.generate(**inputs)

output = tokenizer.decode(outputs[0], skip_special_tokens=True)
output


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

'What is the capital of Australia? Canberra is the capital of Australia. Canberra'

In [18]:
inputs

{'input_ids': tensor([[    1,  1724,   338,   278,  7483,   310,  8314, 29973]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}

In [19]:
outputs

tensor([[    1,  1724,   338,   278,  7483,   310,  8314, 29973,  1815,   495,
           336,   338,   278,  7483,   310,  8314, 29889,  1815,   495,   336]])

In [13]:
from transformers import LlamaForCausalLM, LlamaTokenizer

tokenizer = LlamaTokenizer.from_pretrained("/mnt/chinese-alpaca-2-7b")
model = LlamaForCausalLM.from_pretrained("/mnt/chinese-alpaca-2-7b")

inputs = tokenizer("What is the capital of Australia?", return_tensors="pt")
outputs = model.generate(**inputs)
output = tokenizer.decode(outputs[0], skip_special_tokens=True)
output

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/llama/jupyter/jup_notebook/lib/python3.10/site-packages/transformers/generation/utils.py:1254: UserWarning: Using the model-agnostic default `max_length` (=20) to control thegeneration length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


'What is the capital of Australia? Canberra is the capital of Australia. Canberra'

## 单轮推理

In [1]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    AutoTokenizer,
)

device = "auto"
model_path = "/mnt/chinese-alpaca-2-7b"     # Path to the combined weights

# Prompt should be in this style due to how the data was created
prompt = "#### Human: 上海有什么好吃的东西?#### Assistant:"

bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype="float16",
        bnb_4bit_use_double_quant=True,
    )

model = AutoModelForCausalLM.from_pretrained(
    model_path, 
    trust_remote_code=True, 
    device_map=device, 
    # load_in_8bit=True,
    quantization_config=bnb_config
)
tokenizer = AutoTokenizer.from_pretrained(model_path)

inputs = tokenizer(prompt, return_tensors="pt")
if device != "cpu":
    inputs = inputs.to('cuda')
#del inputs['token_type_ids']

output = model.generate(**inputs, do_sample=True, temperature = 0.2, top_p=0.95, top_k=60, max_new_tokens=500)

'''------------------------------------------------------------------
这个推理函数通过调整不同的参数来生成文本。下面是对这些参数的定义：
1. **inputs：这是一个字典类型的参数，其中包含了模型输入的各种信息，如输入的文本、编码形式等。这些信息通常包括"input_ids"、"attention_mask"和"token_type_ids"等。
2. do_sample：这是一个布尔型参数，默认为True。当设置为True时，模型将使用采样的方式生成文本，即根据概率分布随机选择下一个词。这有助于生成更加多样化和创造性的文本。
------------------------------------------------------------------'''

output = tokenizer.decode(output[0], skip_special_tokens=True)

'''------------------------------------------------------------------
<s> #### Assistant: Saint Bernadette Soubirous
设置了参数skip_special_tokens=True，可以去掉<s>
------------------------------------------------------------------'''

print(output.split("#### Assistant: ")[1])

'''------------------------------------------------------------------
去掉输出文本中的”#### Assistant:”，显示不带任何附加格式的文本。
------------------------------------------------------------------'''


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

上海有很多好吃的东西，以下是一些推荐：
不说废话，直接上菜：
1. 小龙虾：上海的特色美食之一，以新鲜的龙虾为主料，配以各种调料和香料，口感鲜美。
2. 生煎包：上海的传统小吃，外皮酥脆，内馅鲜美，是早餐或下午茶的不错选择。
3. 红烧肉：上海的传统菜肴之一，以猪肉为主料，炖煮至入味，口感鲜嫩。
4. 蟹粉小笼：上海的传统小吃，以鲜蟹肉和蟹黄为主料，蒸制而成，口感鲜美。
5. 红油抄手：上海的传统小吃，以猪肉和虾仁为主料，配以辣椒油和花椒油，口感麻辣。
6. 糯米鸡：上海的传统菜肴之一，以糯米和鸡肉为主料，蒸制而成，口感软糯。
7. 红烧牛肉面：上海的传统菜肴之一，以牛肉为主料，炖煮至入味，口感鲜嫩。
8. 红烧鱼：上海的传统菜肴之一，以鱼为主料，炖煮至入味，口感鲜美。
9. 蟹黄包：上海的传统小吃，以蟹黄为主料，蒸制而成，口感鲜美。
10. 蟹粉汤包：上海的传统小吃，以蟹粉和猪肉为主料，蒸制而成，口感鲜美。


'------------------------------------------------------------------\n去掉输出文本中的”#### Assistant:”，显示不带任何附加格式的文本。\n------------------------------------------------------------------'

In [2]:
inputs

{'input_ids': tensor([[    1,  3191, 12968, 29901, 29871, 32229, 32410, 37253, 32648, 29973,
          4136,  4007, 22137, 29901]], device='cuda:1'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:1')}

In [3]:
output

'#### Human: 上海有什么好吃的东西?#### Assistant: 上海有很多好吃的东西，以下是一些推荐：\n不说废话，直接上菜：\n1. 小龙虾：上海的特色美食之一，以新鲜的龙虾为主料，配以各种调料和香料，口感鲜美。\n2. 生煎包：上海的传统小吃，外皮酥脆，内馅鲜美，是早餐或下午茶的不错选择。\n3. 红烧肉：上海的传统菜肴之一，以猪肉为主料，炖煮至入味，口感鲜嫩。\n4. 蟹粉小笼：上海的传统小吃，以鲜蟹肉和蟹黄为主料，蒸制而成，口感鲜美。\n5. 红油抄手：上海的传统小吃，以猪肉和虾仁为主料，配以辣椒油和花椒油，口感麻辣。\n6. 糯米鸡：上海的传统菜肴之一，以糯米和鸡肉为主料，蒸制而成，口感软糯。\n7. 红烧牛肉面：上海的传统菜肴之一，以牛肉为主料，炖煮至入味，口感鲜嫩。\n8. 红烧鱼：上海的传统菜肴之一，以鱼为主料，炖煮至入味，口感鲜美。\n9. 蟹黄包：上海的传统小吃，以蟹黄为主料，蒸制而成，口感鲜美。\n10. 蟹粉汤包：上海的传统小吃，以蟹粉和猪肉为主料，蒸制而成，口感鲜美。'

## 多轮推理

In [ ]:
import torch
from transformers import (
    LlamaForCausalLM,
    LlamaTokenizer,
    GenerationConfig,
    BitsAndBytesConfig,
)

'''-----------------------------------------------------------------
设置nf4格式的4位量化
-----------------------------------------------------------------'''
bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype="float16",
        bnb_4bit_use_double_quant=True,
    )

'''-----------------------------------------------------------------
将所有推理用的参数放在一起，这样不需要再调用generate函数加很多参数定义
-----------------------------------------------------------------'''
generation_config = GenerationConfig(
    temperature=0.2,
    top_k=40,
    top_p=0.9,
    do_sample=True,
    num_beams=1,
    repetition_penalty=1.1,
    max_new_tokens=400,
    return_dict_in_generate=True,  #返回模型输出而不是普通元组
    #output_scores=False,
    )

'''-----------------------------------------------------------------
判断GPU是否存在，存在的话device指向GPU
-----------------------------------------------------------------'''
if torch.cuda.is_available():
    device = torch.device(0)
else:
    device = torch.device('cpu')

'''-----------------------------------------------------------------
加载词汇表与模型
-----------------------------------------------------------------'''
base_model = "/mnt/chinese-alpaca-2-7b"
tokenizer_path = "/mnt/chinese-alpaca-2-7b"

tokenizer = LlamaTokenizer.from_pretrained(tokenizer_path)

model = LlamaForCausalLM.from_pretrained(
    base_model, 
    load_in_8bit = True,
    torch_dtype = torch.float16,
    low_cpu_mem_usage=True,
    device_map = 'auto',
    quantization_config = bnb_config,
    )

if device==torch.device('cpu'):
    model.float()

model.eval()

'''-----------------------------------------------------------------
将用户输入的提示加入系统提示中，生成实际送给模型推理的提示
-----------------------------------------------------------------'''
def generate_prompt(instruction):
    return f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.
    
### Instruction:
{instruction}

### Response: """

def predict(input, history):
    now_input = input

    '''-----------------------------------------------------------------
    合并缓存中的内容，形成包含上下文的提示
    -----------------------------------------------------------------'''
    history = history or []
    if len(history) != 0:
        input = "".join(["### Instruction:\n" + i[0] +"\n\n" + "### Response: " + i[1] + "\n\n" for i in history]) + \
        "### Instruction:\n" + input
        input = input[len("### Instruction:\n"):]
        if len(input) > max_memory:
            input = input[-max_memory:]
            
    prompt = generate_prompt(input)
    
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs["input_ids"].to(device)
    
    with torch.no_grad():
        generation_output = model.generate(
            input_ids=input_ids,
            generation_config=generation_config,
        )
    s = generation_output.sequences[0]
    output = tokenizer.decode(s, skip_special_tokens=True)
    output = output.split("### Response:")[-1].strip()
    history.append((now_input, output))

    return output, history

history = []

while True:
    input_text = input("Input: ")
    if len(input_text.strip())==0:
        break        
    output, history = predict(input_text, history)
    
    print(f"{output}\n")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Input:  上海有什么好吃的？


上海有很多好吃的食物，比如小笼包、生煎馒头和红烧肉等。



Input:  有什么好玩的？


上海有许多有趣的地方可以参观，例如外滩、城隍庙和豫园等。



Input:  有哪些标志性建筑？


上海的标志性建筑包括东方明珠塔、上海中心大厦和上海环球金融中心等。



In [36]:
history

[('上海有什么好吃的？', '上海有很多好吃的食物，比如小龙虾、生煎馒头和蟹黄包。'),
 ('还有什么好玩的？', '除了美食之外，上海还有许多有趣的活动可以参加，如参观外滩、游览城隍庙或去豫园游玩。'),
 ('有哪些标志性建筑？', '上海有许多标志性的建筑，例如东方明珠塔、上海中心大厦以及上海环球金融中心等。')]

## Gradio 多轮推理

In [3]:
import gradio as gr
import torch
from transformers import LlamaForCausalLM, LlamaTokenizer, GenerationConfig
from peft import PeftModel

generation_config = dict(
    temperature=0.2,
    top_k=40,
    top_p=0.9,
    do_sample=True,
    num_beams=1,
    repetition_penalty=1.1,
    max_new_tokens=400
    )

load_type = torch.float16

if torch.cuda.is_available():
    device = torch.device(0)
else:
    device = torch.device('cpu')
    
base_model = "/mnt/chinese-alpaca-2-7b"
tokenizer_path = "/mnt/chinese-alpaca-2-7b"
lora_model = None

tokenizer = LlamaTokenizer.from_pretrained(tokenizer_path)

base_model = LlamaForCausalLM.from_pretrained(
    base_model, 
    load_in_8bit=True,
    torch_dtype=load_type,
    low_cpu_mem_usage=True,
    device_map='auto',
    )

if lora_model is not None:
    print("loading peft model")
    model = PeftModel.from_pretrained(base_model, args.lora_model,torch_dtype=load_type,device_map='auto',)
else:
    model = base_model

if device==torch.device('cpu'):
    model.float()

model.eval()

def reset_user_input():
    return gr.update(value='')

def reset_state():
    return [], []

def generate_prompt(instruction):
    return f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.
    
### Instruction:
{instruction}

### Response: """

def predict(
    input,
    chatbot,
    history,
    max_new_tokens=128,
    top_p=0.75,
    temperature=0.1,
    top_k=40,
    num_beams=4,
    repetition_penalty=1.0,
    max_memory=256,
    **kwargs,
):
    now_input = input
    chatbot.append((input, ""))
    history = history or []
    if len(history) != 0:
        input = "".join(["### Instruction:\n" + i[0] +"\n\n" + "### Response: " + i[1] + "\n\n" for i in history]) + \
        "### Instruction:\n" + input
        input = input[len("### Instruction:\n"):]
        if len(input) > max_memory:
            input = input[-max_memory:]
    prompt = generate_prompt(input)
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs["input_ids"].to(device)
    generation_config = GenerationConfig(
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        num_beams=num_beams,
        **kwargs,
    )
    with torch.no_grad():
        generation_output = model.generate(
            input_ids=input_ids,
            generation_config=generation_config,
            return_dict_in_generate=True,
            output_scores=False,
            max_new_tokens=max_new_tokens,
            repetition_penalty=float(repetition_penalty),
        )
    s = generation_output.sequences[0]
    output = tokenizer.decode(s, skip_special_tokens=True)
    output = output.split("### Response:")[-1].strip()
    history.append((now_input, output))
    chatbot[-1] = (now_input, output)
    return chatbot, history

with gr.Blocks() as demo:
    gr.HTML("""<h1 align="center">Chinese LLaMA & Alpaca LLM</h1>""")
    gr.Markdown("> 为了促进大模型在中文NLP社区的开放研究，本项目开源了中文LLaMA模型和指令精调的Alpaca大模型。这些模型在原版LLaMA的基础上扩充了中文词表并使用了中文数据进行二次预训练，进一步提升了中文基础语义理解能力。同时，中文Alpaca模型进一步使用了中文指令数据进行精调，显著提升了模型对指令的理解和执行能力")
    chatbot = gr.Chatbot()
    with gr.Row():
        with gr.Column(scale=4):
            with gr.Column(scale=12):
                user_input = gr.Textbox(show_label=False, placeholder="Input...", lines=10).style(
                    container=False)
            with gr.Column(min_width=32, scale=1):
                submitBtn = gr.Button("Submit", variant="primary")
        with gr.Column(scale=1):
            emptyBtn = gr.Button("Clear History")
            max_length = gr.Slider(
                0, 4096, value=128, step=1.0, label="Maximum length", interactive=True)
            top_p = gr.Slider(0, 1, value=0.8, step=0.01,
                              label="Top P", interactive=True)
            temperature = gr.Slider(
                0, 1, value=0.7, step=0.01, label="Temperature", interactive=True)

    history = gr.State([])  # (message, bot_message)

    submitBtn.click(predict, [user_input, chatbot, history, max_length, top_p, temperature], [chatbot, history],
                    show_progress=True)
    submitBtn.click(reset_user_input, [], [user_input])

    emptyBtn.click(reset_state, outputs=[chatbot, history], show_progress=True)
    
demo.queue().launch(share=True, inbrowser=True, server_name = '0.0.0.0', server_port=19324)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/tmp/ipykernel_100834/2372455135.py:117: GradioDeprecationWarning: The `style` method is deprecated. Please set these arguments in the constructor instead.
  user_input = gr.Textbox(show_label=False, placeholder="Input...", lines=10).style(


Running on local URL:  http://0.0.0.0:19324

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


## 使用vLLM

In [ ]:
import torch
from transformers import LlamaForCausalLM, LlamaTokenizerFast
from vllm import LLM, SamplingParams

DEFAULT_SYSTEM_PROMPT = """You are a helpful assistant. 你是一个乐于助人的助手。"""

TEMPLATE = (
    "[INST] <<SYS>>\n"
    "{system_prompt}\n"
    "<</SYS>>\n\n"
    "{instruction} [/INST]"
)

generation_config = dict(
    temperature=0.2,
    top_k=40,
    top_p=0.9,
    max_tokens=400,
    presence_penalty=1.0,
)

sample_data = ["为什么要减少污染，保护环境？"]
system_prompt = DEFAULT_SYSTEM_PROMPT
with_prompt = True
gpus = "0,1"

def generate_prompt(instruction, system_prompt=DEFAULT_SYSTEM_PROMPT):
    return TEMPLATE.format_map({'instruction': instruction,'system_prompt': system_prompt})

if __name__ == '__main__':
    load_type = torch.float16
    
    if torch.cuda.is_available():
        device = torch.device(0)
    else:
        device = torch.device('cpu')

    base_model = "/mnt/chinese-alpaca-2-7b"
    tokenizer_path = "/mnt/chinese-alpaca-2-7b"
    lora_model = None

    model = LLM(model = base_model,
        tokenizer = tokenizer_path,
        tokenizer_mode = 'auto',
        tensor_parallel_size = len(gpus.split(',')))
    tokenizer = LlamaTokenizerFast.from_pretrained(tokenizer_path, legacy=True)

    with torch.no_grad():
        print("Start inference with instruction mode.")

        print('='*85)
        print("+ 该模式下仅支持单轮问答，无多轮对话能力。\n"
              "+ 如要进行多轮对话，请使用llama.cpp或本项目中的gradio_demo.py。")
        print('-'*85)
        print("+ This mode only supports single-turn QA.\n"
              "+ If you want to experience multi-turn dialogue, please use llama.cpp or gradio_demo.py.")
        print('='*85)

        while True:
            raw_input_text = input("Input:")
            if len(raw_input_text.strip())==0:
                break
            if with_prompt:
                input_text = generate_prompt(instruction = raw_input_text, system_prompt = system_prompt)
            else:
                input_text = raw_input_text

            output = model.generate([input_text], SamplingParams(**generation_config), use_tqdm=False)
            response = output[0].outputs[0].text

            print("Response: ",response)
            print("\n")


## 解码策略

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

device = "auto"
model_path = "/mnt/chinese-alpaca-2-7b"     # Path to the combined weights

bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype="float16",
        bnb_4bit_use_double_quant=True,
    )

model = AutoModelForCausalLM.from_pretrained(
    model_path, 
    trust_remote_code=True, 
    device_map=device, 
    quantization_config=bnb_config
)
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [ ]:
def sample_from_model(model, input_ids, temperature=1.0, top_k=0, top_p=0.0):
    with torch.no_grad():
        outputs = model(input_ids = input_ids)
        logits = outputs.logits        
        logits = logits[:, -1, :]  # get the logits for the last predicted token
        
        if temperature > 0.0:
            logits /= temperature

        filtered_logits = top_k_top_p_filtering(logits, top_k=top_k, top_p=top_p)
        probabilities = F.softmax(filtered_logits, dim=-1)
        next_token = torch.multinomial(probabilities, num_samples=1)
    return next_token

def top_k_top_p_filtering(logits, top_k=0, top_p=0.0, filter_value=-float('Inf')):
    top_k = min(top_k, logits.size(-1))  # Safety check
    
    logits = logits.squeeze()   #在张量中删除维度为1,减少张量的维度。

    if top_k > 0:
        indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
        logits[indices_to_remove] = filter_value
 
    if top_p > 0.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = 0

        indices_to_remove = sorted_indices[sorted_indices_to_remove]
        logits[indices_to_remove] = filter_value
    return logits

prompt = "我下午要去" 
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"]

temperature = 0.8
top_k = 50
top_p = 0.9

next_token = sample_from_model(model, input_ids, temperature=temperature, top_k=top_k, top_p=top_p)

print(next_token)
print(tokenizer.decode(next_token))

In [3]:
indices = [31811, 32322, 31026, 34133, 32074, 49717, 35265, 32227, 32175, 41669, 32002, 39455, 32334, 34916, 32424, 32215, 47006, 34204, 32429, 31656, 32229, 32019, 34061, 31999, 32139, 32492, 31092, 37408, 32201, 40287, 30415, 35246, 49007, 43117]
for t in indices:
    print("%s,"%(tokenizer.decode(t)), end='')

看,参加,开,面试,买,拜访,上课,找,学校,逛街,一个,看电影,医院,购物,考试,见,一趟,上班,办,打,上海,做,机场,给,北京,喝,接,参观,听,健身房,学,超市,打篮球,散步,